In [ ]:
from lets_plot import *

LetsPlot.setup_html()
import polars as pl

In [ ]:
frame = pl.read_csv("benchmarks/results/results.csv")
frame

In [ ]:
# fr = fr.filter(pl.col("phase") != "total")

In [ ]:
fr = frame

In [ ]:
ratios = (
    fr.group_by(["dataset", "framework"])
    .agg(pl.col("time_median_s").median().alias("t"))
    .pivot(on="framework", index="dataset", values="t")
    .with_columns(speedup=(pl.col("scanpy") / pl.col("cellestial")))
    .with_columns(label=pl.format("{}x", pl.col("speedup").round(2)))
)

y_top = fr["time_median_s"].max() * 0.8  # headroom above tallest box (log scale)

(
    ggplot(fr)
    + geom_boxplot(aes(x="dataset", y="time_median_s", fill="framework"))
    + geom_text(aes(x="dataset", label="label"), y=y_top, data=ratios, size=10, color="orange")
    + scale_y_log10()
) + ggsize(800, 800)


In [ ]:
(
    ggplot(fr)
    + geom_boxplot(
        aes(x="dataset", y="time_median_s", fill="framework"),
    )
    + scale_y_log10()
)

In [ ]:
(
    ggplot(fr)
    + geom_boxplot(
        aes(x="case", y="time_median_s", color="framework"),
        trim=False,
        scale="width",
    )
    + scale_y_log10()
) + facet_wrap("phase")

In [ ]:
non_html = frame.filter( ~pl.col("phase").str.contains("html"))

In [ ]:
(
    ggplot(non_html)
    + geom_point(aes(x="n_cells", y="time_median_s", color="framework", shape="case"))
    + scale_y_log10()
    + geom_smooth(aes(x="n_cells", y="time_median_s", color="framework"))
    + facet_wrap("phase")
)

In [ ]:
(
    ggplot(fr)
    + geom_boxplot(
        aes(x="framework", y="time_median_s", color="phase", fill="framework"),
        trim=False,
        scale="width",
    )
    + scale_y_log10()
)

In [ ]:
fr = fr.filter(pl.col("peak_mem_mb") > 0)
(
    ggplot(fr)
    + geom_boxplot(
        aes(x="case", y="peak_mem_mb", color="framework"),
        trim=False,
        scale="width",
    )
    + scale_y_log10()
) + facet_wrap("phase")

In [ ]:
fr

In [ ]:
render_only = frame.filter(pl.col("phase").str.contains("render"))
render_only

In [ ]:
render_only = render_only.with_columns(
    render_type=pl.when(pl.col("phase").str.contains("html")).then(pl.lit("html")).otherwise(pl.lit("svg"))
)

In [ ]:
(
    ggplot(render_only)
    + geom_boxplot(
        aes(x="case", y="time_median_s", color="framework", fill="render_type"),
        trim=False,
        scale="width",
        tooltips=["render_type"],

    ) + ggsize(800, 800) + scale_fill_viridis()
)